# Project: a CNN reads your digits
**Machine Learning · Week 9 (Deep learning: images, sequences and attention) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w9d3.html

In week 5, a plain network read the $8 \times 8$ digits with 97% accuracy. But every digit in that dataset
is neatly centered. What happens when one is written a little to the side? Today: a CNN against the plain
network, on digits that move.

In [ ]:
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
digits = load_digits()
images = digits.images / 16.0                 # (1797, 8, 8), values 0 to 1
labels = digits.target
X_rest, X_test, y_rest, y_test = train_test_split(images, labels, test_size=0.2, random_state=0, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)

def shift(imgs, dx, dy):
    """Move a stack of images dx pixels right and dy pixels down; what moves out is lost, blanks move in."""
    out = np.roll(imgs, (dy, dx), axis=(1, 2))
    if dy > 0: out[:, :dy, :] = 0
    if dy < 0: out[:, dy:, :] = 0
    if dx > 0: out[:, :, :dx] = 0
    if dx < 0: out[:, :, dx:] = 0
    return out

fig, axes = plt.subplots(1, 4, figsize=(8, 2.4), layout="constrained")
for ax, (dx, dy) in zip(axes, [(0, 0), (1, 0), (2, 0), (1, 1)]):
    ax.imshow(shift(X_test[:1], dx, dy)[0], cmap="Purples", vmin=0, vmax=1)
    ax.set_title(f"moved ({dx}, {dy})", fontsize=11)
    ax.axis("off")
plt.show()

## The contestants
The plain network from week 5, and a small CNN: three convolution layers with one pooling step, then a
**global max pool** that keeps only the strongest response of each of the 32 channels, wherever it is.

In [ ]:
def make_mlp():
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

def make_cnn():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 8 x 8 -> 4 x 4
        nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),                             # the strongest response per channel
        nn.Linear(32, 10))

count = lambda model: sum(p.numel() for p in model.parameters())
print(f"plain network: {count(make_mlp()):,} weights")
print(f"CNN:           {count(make_cnn()):,} weights")

## Train both, then test on moved digits
The same habits as week 5: mini-batches, Adam, and keeping the weights with the best validation loss.
`augment=True` will come in handy in a minute.

In [ ]:
T = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(1)          # (n, 1, 8, 8)
L = lambda a: torch.tensor(a, dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()

def fit(model, augment=False, epochs=60):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    rng = np.random.default_rng(0)
    best_loss, best_state = float("inf"), None
    for epoch in range(epochs):
        order = rng.permutation(len(X_train))
        for i in range(0, len(order), 32):
            idx = order[i:i + 32]
            xb = X_train[idx]
            if augment:                                    # move each image by -1, 0 or +1 pixels each way
                xb = np.stack([shift(xb[j:j + 1], *rng.integers(-1, 2, 2))[0] for j in range(len(xb))])
            opt.zero_grad()
            loss_fn(model(T(xb)), L(y_train[idx])).backward()
            opt.step()
        with torch.no_grad():
            val_loss = loss_fn(model(T(X_val)), L(y_val)).item()
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model

def accuracy(model, imgs, y):
    with torch.no_grad():
        return (model(T(imgs)).argmax(dim=1).numpy() == y).mean()

def shift_test(model):
    """Test accuracy with every test digit moved by 0, 1 and 2 pixels to the right."""
    return [accuracy(model, shift(X_test, dx, 0), y_test) for dx in (0, 1, 2)]

t0 = time.perf_counter()
mlp, cnn = fit(make_mlp()), fit(make_cnn())
print(f"trained both in {time.perf_counter() - t0:.0f} s\n")
print(f"{'':14s} {'centered':>9s} {'moved 1':>8s} {'moved 2':>8s}")
for name, model in [("plain network", mlp), ("CNN", cnn)]:
    print(f"{name:14s}" + "".join(f"{a:9.3f}" for a in shift_test(model)))

## Teach it that position doesn't matter: data augmentation
The CNN copes better, but both suffer. Fix: during training, show every digit at a random small shift.
That's **data augmentation**: more varied examples for free. Same models, same loop, `augment=True`.

In [ ]:
t0 = time.perf_counter()
mlp_aug, cnn_aug = fit(make_mlp(), augment=True), fit(make_cnn(), augment=True)
print(f"trained both in {time.perf_counter() - t0:.0f} s\n")
results = {"plain network": shift_test(mlp), "CNN": shift_test(cnn),
           "plain network + augmentation": shift_test(mlp_aug), "CNN + augmentation": shift_test(cnn_aug)}
plt.figure(figsize=(6.4, 3.6))
for (name, acc), color, dash in zip(results.items(), ["#ff8a50", "#17a98c", "#ff8a50", "#17a98c"], ["--", "--", "-", "-"]):
    plt.plot([0, 1, 2], acc, dash, marker="o", color=color, label=name)
plt.xticks([0, 1, 2])
plt.xlabel("pixels moved to the right")
plt.ylabel("test accuracy")
plt.ylim(0, 1.02)
plt.legend(fontsize=9.5)
plt.title("Trained on shifts of at most 1 pixel")
plt.show()
print(f"{'':30s} {'centered':>9s} {'moved 1':>8s} {'moved 2':>8s}")
for name, acc in results.items():
    print(f"{name:30s}" + "".join(f"{a:9.3f}" for a in acc))

## Your turn: which way hurts most?
Test `cnn_aug` and `mlp_aug` on test digits moved by 2 pixels in each of four directions: right `(2, 0)`,
left `(-2, 0)`, down `(0, 2)` and up `(0, -2)`. Is every direction equally hard? Why might it not be?

In [ ]:
for dx, dy in [(2, 0), (-2, 0), (0, 2), (0, -2)]:
    moved = None    # TODO: shift(X_test, dx, dy)
    print(f"moved ({dx:+d}, {dy:+d}):  plain network {None}   CNN {None}")   # TODO: accuracy(...) of both

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for dx, dy in [(2, 0), (-2, 0), (0, 2), (0, -2)]:
    moved = shift(X_test, dx, dy)
    print(f"moved ({dx:+d}, {dy:+d}):  plain network {accuracy(mlp_aug, moved, y_test):.3f}   CNN {accuracy(cnn_aug, moved, y_test):.3f}")
print("The digits fill the full height of the 8 x 8 box but not its width, so moving up or down")
print("pushes part of the digit out of the picture: no network can read ink that isn't there.")